# Weeks 1–2 · Exercises

Fill in each function so the **check cell** below it passes without an `AssertionError`.
Work in order; later exercises reuse earlier ones.
The solutions are in `solutions/02_solutions.ipynb` — look only after you have tried.

In [ ]:
import csv, time
from pathlib import Path
from functools import wraps
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())

### Exercise 1 · Comprehensions

Given a list of `(name, gpa)` pairs, return a dict mapping each name whose GPA is at least 3.0 to its GPA **scaled to a percentage** (GPA/4 × 100, rounded to one decimal).

In [ ]:
def honours_pct(pairs):
    ...

In [ ]:
assert honours_pct([("a", 3.0), ("b", 2.9), ("c", 4.0)]) == {"a": 75.0, "c": 100.0}
assert honours_pct([]) == {}
print("✓ exercise 1")

### Exercise 2 · Flatten

Return a flat list from a list of lists **of lists** (two levels), e.g. `[[[1],[2,3]],[[4]]]` → `[1,2,3,4]`. One comprehension.

In [ ]:
def flatten2(nested):
    ...

In [ ]:
assert flatten2([[[1], [2, 3]], [[4]]]) == [1, 2, 3, 4]
assert flatten2([[[]]]) == []
print("✓ exercise 2")

### Exercise 3 · Generator over a file

Write a generator `warnings(path)` that yields the **user id (as int)** of every `WARNING` line in `server.log`, without loading the file into a list.

In [ ]:
def warnings(path):
    ...

In [ ]:
import types
g = warnings(DATA / "server.log")
assert isinstance(g, types.GeneratorType)
ids = list(g)
assert all(isinstance(i, int) for i in ids) and 50 < len(ids) < 200, len(ids)
print("✓ exercise 3 -", len(ids), "warnings")

### Exercise 4 · Lazy running mean

Write a generator `running_mean(values)` that yields the mean of all values seen so far after each value. It must work for an infinite input, so use no `list()`/`sum()` over the whole input.

In [ ]:
def running_mean(values):
    ...

In [ ]:
from itertools import count, islice
assert list(running_mean([2, 4, 6])) == [2.0, 3.0, 4.0]
assert list(islice(running_mean(count(1)), 4)) == [1.0, 1.5, 2.0, 2.5]   # infinite input
print("✓ exercise 4")

### Exercise 5 · Sorting with keys

Sort a list of `(name, major, gpa)` tuples by major ascending, then GPA **descending**, then name ascending. Return the new list.

In [ ]:
def rank(rows):
    ...

In [ ]:
rows = [("Omar", "CS", 3.1), ("Sara", "DS", 3.9), ("Ahmad", "CS", 3.1), ("Lina", "CS", 3.8)]
assert rank(rows) == [("Lina", "CS", 3.8), ("Ahmad", "CS", 3.1), ("Omar", "CS", 3.1), ("Sara", "DS", 3.9)]
print("✓ exercise 5")

### Exercise 6 · A counting decorator

Write a decorator `count_calls` that adds a `.calls` attribute to the wrapped function, incremented on every call, while preserving the function's name and return value.

In [ ]:
def count_calls(func):
    ...

In [ ]:
@count_calls
def square(x):
    """doc"""
    return x * x
assert [square(2), square(3)] == [4, 9]
assert square.calls == 2 and square.__name__ == "square" and square.__doc__ == "doc"
print("✓ exercise 6")

### Exercise 7 · Safe parsing

`to_float(s, default=None)` returns `float(s)` — also accepting a **comma decimal separator** like `"3,1"` — and `default` for anything unparseable, including `None`.

In [ ]:
def to_float(s, default=None):
    ...

In [ ]:
assert to_float("3.5") == 3.5 and to_float("3,1") == 3.1
assert to_float("N/A") is None and to_float("", default=0.0) == 0.0 and to_float(None, -1) == -1
print("✓ exercise 7")

### Exercise 8 · Typed record + validation

Create a `@dataclass` `Course(code: str, title: str, credits: int)` whose `__post_init__` raises `ValueError` if `credits` is not between 1 and 6, or if `code` is not exactly six digits.

In [ ]:
from dataclasses import dataclass

@dataclass
class Course:
    ...

In [ ]:
c = Course("606315", "Programming for DS", 3)
assert (c.code, c.credits) == ("606315", 3)
for bad in [("60631", "x", 3), ("606315", "x", 0), ("606315", "x", 7)]:
    try:
        Course(*bad); raise AssertionError(f"should have failed: {bad}")
    except ValueError:
        pass
print("✓ exercise 8")

### Exercise 9 · Context manager

Write a context manager `cd(path)` (use `contextlib.contextmanager`) that changes the working directory on entry and **always** restores it on exit, even after an exception.

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def cd(path):
    ...

In [ ]:
import os
start = os.getcwd()
with cd(".."):
    assert os.getcwd() != start
assert os.getcwd() == start
try:
    with cd(".."):
        raise RuntimeError("boom")
except RuntimeError:
    pass
assert os.getcwd() == start
print("✓ exercise 9")

### Exercise 10 · Mini ETL

Using only the standard library, read `students.csv` lazily and return a dict `{major: (count, mean_gpa)}` with majors **stripped of whitespace**, unparseable GPAs skipped, and means rounded to 2 decimals. Reuse `to_float` from Exercise 7.

In [ ]:
def gpa_by_major(path):
    ...

In [ ]:
res = gpa_by_major(DATA / "students.csv")
assert set(res) == {"Data Science", "Computer Science", "Cybersecurity", "AI", "Software Eng"}, set(res)
assert all(isinstance(v, tuple) and len(v) == 2 for v in res.values())
assert sum(c for c, _ in res.values()) > 180
assert all(1.5 <= m <= 4.0 for _, m in res.values())
print("✓ exercise 10")
for k, (n, m) in sorted(res.items()): print(f"  {k:18s} n={n:3d} mean={m}")